# 01. FBref Data Ingestion

**Stage:** Ingestion  
**Inputs:** `soccerdata` FBref and ClubElo readers, EPL seasons from config  
**Outputs:** `data/raw/{season}/fixtures.parquet`, `data/raw/{season}/match_stats.parquet`

This notebook loads fixtures and team match statistics. Player availability is deferred to Task 2.1 because retrieving player match pages one fixture at a time is expensive and minutes alone cannot prove an absence reason.

In [1]:
# Load configuration and imports
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config.loader import load_config
from src.ingestion.soccerdata_client import SoccerDataClient, SoccerDataConfig

config = load_config()
season = config["data"]["seasons"][0]
league_id = config["data"]["default_league_id"]
ingestion_config = config["ingestion"]["soccerdata"]
print(f"Season: {season}")
print(f"League ID: {league_id}")
fixtures = None
match_stats = None


[08/21/26 01:20:38] INFO     No custom team name replacements found. You can configure these in       ]8;id=3845005;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=3845006;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/mac/soccerdata/config/teamname_replacements.json.                              

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=3845012;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=3845013;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/mac/soccerdata/config/league_dict.json.                                        

Season: 2024/2025
League ID: 47


In [2]:
# NBVAL_SKIP
client = SoccerDataClient(config=SoccerDataConfig(league=ingestion_config["league"]))
try:
    fixtures = client.fetch_fixtures(season=season, league_id=league_id)
    print(f"Fetched {len(fixtures)} fixtures from FBref")
except Exception as exc:
    print(f"Error fetching fixtures: {exc}")
    fixtures = None

[08/21/26 01:20:41] INFO     Saving cached data to /Users/mac/soccerdata/data/FBref                  ]8;id=3845020;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=3845021;file:///Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/soccerdata/_common.py#250\250]8;;\

Fetched 380 fixtures from FBref


In [3]:
# NBVAL_SKIP
# Fetch team match statistics once; soccerdata returns the full season table.
try:
    if fixtures is not None and len(fixtures) > 0:
        match_stats = client.fetch_all_match_stats(season=season)
        print(f"Fetched {len(match_stats)} team match-stat rows")
    else:
        match_stats = None
except Exception as exc:
    print(f"Error fetching match stats: {exc}")
    match_stats = None

Fetched 760 team match-stat rows


In [4]:
# Player availability is deferred to Task 2.1.
# Do not retrieve player match pages for every fixture during base ingestion.
player_minutes = None
print("Player availability retrieval deferred to the absence-classification stage.")

Player availability retrieval deferred to the absence-classification stage.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Clean season string for safe directory naming
season_clean = season.replace("/", "-")

# Construct the absolute path directly from the root
output_dir = PROJECT_ROOT / "data" / "raw" / season_clean
output_dir.mkdir(parents=True, exist_ok=True)

if fixtures is not None and len(fixtures) > 0:
    fixtures.to_parquet(output_dir / "fixtures.parquet", index=False)
    print(f"Saved {len(fixtures)} fixtures")
if match_stats is not None and len(match_stats) > 0:
    match_stats.to_parquet(output_dir / "match_stats.parquet", index=False)
    print(f"Saved {len(match_stats)} team match-stat rows")

../../data/raw/2024-2025
Saved 380 fixtures
Saved 760 team match-stat rows


In [6]:
if fixtures is not None and len(fixtures) > 0:
    print("Sample fixtures:")
    display(fixtures.head())
if match_stats is not None and len(match_stats) > 0:
    print("Sample team match statistics:")
    display(match_stats.head())

Sample fixtures:


,league,season,game,week,day,date,time,home_team,score,away_team,attendance,venue,referee,match_report,notes,fixture_id,home_goals,away_goals
0,ENG-Premier League,2425,2024-08-16 Manchester Utd-Fulham,1,Fri,2024-08-16,20:00,Manchester Utd,1–0,Fulham,73297,Old Trafford,Robert Jones,/en/matches/cc5b4244/Manchester-United-Fulham-...,<NA>,cc5b4244,1.0,0.0
1,ENG-Premier League,2425,2024-08-17 Arsenal-Wolves,1,Sat,2024-08-17,15:00,Arsenal,2–0,Wolves,60261,Emirates Stadium,Jarred Gillett,/en/matches/c0e3342a/Arsenal-Wolverhampton-Wan...,<NA>,c0e3342a,2.0,0.0
2,ENG-Premier League,2425,2024-08-17 Everton-Brighton,1,Sat,2024-08-17,15:00,Everton,0–3,Brighton,39217,Goodison Park,Simon Hooper,/en/matches/71618ace/Everton-Brighton-and-Hove...,<NA>,71618ace,0.0,3.0
3,ENG-Premier League,2425,2024-08-17 Ipswich Town-Liverpool,1,Sat,2024-08-17,12:30,Ipswich Town,0–2,Liverpool,30014,Portman Road Stadium,Tim Robinson,/en/matches/a1d0d529/Ipswich-Town-Liverpool-Au...,<NA>,a1d0d529,0.0,2.0
4,ENG-Premier League,2425,2024-08-17 Newcastle-Southampton,1,Sat,2024-08-17,15:00,Newcastle,1–0,Southampton,52196,St James' Park,Craig Pawson,/en/matches/34557647/Newcastle-United-Southamp...,<NA>,34557647,1.0,0.0


Sample team match statistics:


,league,season,team,game,date,time,round,day,venue,result,...,GA,opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,match_report,Notes
0,ENG-Premier League,2425,NaN,2024-08-16 Manchester Utd-nan,2024-08-16,20:00:00,Matchweek 1,Fri,Away,L,...,1,Manchester Utd,45,73297,Bernd Leno,4-2-3-1,4-2-3-1,Robert Jones,/en/matches/cc5b4244/Manchester-United-Fulham-...,<NA>
1,ENG-Premier League,2425,NaN,2024-08-16 nan-Fulham,2024-08-16,20:00:00,Matchweek 1,Fri,Home,W,...,0,Fulham,55,73297,Bruno Fernandes,4-2-3-1,4-2-3-1,Robert Jones,/en/matches/cc5b4244/Manchester-United-Fulham-...,<NA>
2,ENG-Premier League,2425,NaN,2024-08-17 Arsenal-nan,2024-08-17,15:00:00,Matchweek 1,Sat,Away,L,...,2,Arsenal,47,60261,Mario Lemina,4-2-3-1,4-3-3,Jarred Gillett,/en/matches/c0e3342a/Arsenal-Wolverhampton-Wan...,<NA>
3,ENG-Premier League,2425,NaN,2024-08-17 Everton-nan,2024-08-17,15:00:00,Matchweek 1,Sat,Away,W,...,0,Everton,63,39217,Lewis Dunk,4-2-3-1,4-2-3-1,Simon Hooper,/en/matches/71618ace/Everton-Brighton-and-Hove...,<NA>
4,ENG-Premier League,2425,NaN,2024-08-17 Ipswich Town-nan,2024-08-17,12:30:00,Matchweek 1,Sat,Away,W,...,0,Ipswich Town,62,30014,Virgil van Dijk,4-2-3-1,4-2-3-1,Tim Robinson,/en/matches/a1d0d529/Ipswich-Town-Liverpool-Au...,<NA>


In [7]:
print("=== Data Validation ===")
if fixtures is not None and len(fixtures) > 0:
    required_columns = ["fixture_id", "home_team", "away_team", "date"]
    missing_columns = [column for column in required_columns if column not in fixtures.columns]
    print(f"Fixtures: {len(fixtures)} rows")
    print(f"Missing required columns: {missing_columns}")
if match_stats is not None and len(match_stats) > 0:
    print(f"Team match stats: {len(match_stats)} rows")
print("=== Ingestion Complete ===")

=== Data Validation ===
Fixtures: 380 rows
Missing required columns: []
Team match stats: 760 rows
=== Ingestion Complete ===
